# DE-4) library `pandas` สำหรับงาน Data Engineer เบื้องต้น

สำหรับ Data Engineer, `pandas` ไม่ได้มีไว้แค่ “วิเคราะห์ข้อมูล”  
แต่เป็นเครื่องมือสำคัญในการทำงานแบบ pipeline เช่น

- อ่านไฟล์ CSV/TSV/Excel จากหลายแหล่ง
- ตรวจสอบคุณภาพข้อมูล (data validation เบื้องต้น)
- แปลง/ทำความสะอาดข้อมูล (data transformation)
- รวมหลายไฟล์เข้าด้วยกัน (batch ingestion)
- ส่งต่อเป็นไฟล์/รูปแบบที่ระบบอื่นใช้

**หัวใจของบทนี้:** เข้าใจการตั้งค่า (configuration) ของ `pd.read_csv()` และการเขียนไฟล์อย่างถูกต้อง  
เพราะในงานจริง “ไฟล์ไม่เคยสวย” และ “ค่า default มักไม่พอ”

---

## สิ่งที่คุณจะได้จากบทนี้
- รู้จัก DataFrame ในมุม Data Engineer (ตารางข้อมูลเพื่อ ETL/ELT)
- ใช้ `pd.read_csv()` และเข้าใจพารามิเตอร์สำคัญที่ต้องใช้บ่อย
- ตรวจสอบข้อมูลเบื้องต้น: shape, columns, dtypes, missing values
- แปลงข้อมูลพื้นฐานที่ใช้ใน pipeline
- เขียนไฟล์ออกด้วย `to_csv()` พร้อมพารามิเตอร์สำคัญ

---

## ภาพรวมเนื้อหา (ลำดับการเรียน)
1. ติดตั้ง/นำเข้า pandas
2. สร้างไฟล์ตัวอย่าง (จำลองไฟล์จากระบบจริง)
3. `pd.read_csv()` เบื้องต้น + ทำความเข้าใจ output
4. พารามิเตอร์สำคัญของ `pd.read_csv()` (DE focus)
5. Data quality check เบื้องต้น
6. Transformation เบื้องต้นใน pipeline
7. `to_csv()` และพารามิเตอร์สำคัญ
8. Mini pipeline: อ่านหลายไฟล์ → รวม → ทำความสะอาด → เขียนผลลัพธ์

## 1) ติดตั้งและ import pandas

✅ หากยังไม่มี pandas ให้ติดตั้ง (อาจใช้เวลาสักครู่)

In [1]:
!pip -q install pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import pandas as pd
pd.__version__

'3.0.0'

## 2) สร้างไฟล์ CSV ตัวอย่าง (จำลองไฟล์จากระบบจริง)

เราจะสร้างไฟล์ที่มี “ประเด็นที่เจอบ่อย” เช่น
- มีค่าว่าง (missing)
- ตัวเลขปะปนข้อความ
- วันที่เป็นข้อความ

เพื่อฝึกการตั้งค่าและตรวจสอบ

In [3]:
import os

os.makedirs("pandas_data", exist_ok=True)

raw_path = os.path.join("pandas_data", "raw_sales.csv")

raw_text = """date,store_id,sales,comment
2025-01-01,001,1200,ok
2025-01-02,001,,
2025-01-03,002,1500,good
2025-01-04,002,1,500,bad_comma
2025-01-05,003,900,ok
"""

# หมายเหตุ: แถวที่ 2025-01-04 มี comma ในตัวเลข 1,500 ซึ่งทำให้ CSV "พัง" ได้ในโลกจริง
# เพื่อให้ไฟล์อ่านได้ เราจะทำไฟล์ที่ถูกต้องอีกไฟล์สำหรับทดลองหลัก และเก็บเคสพังไว้เป็นบทเรียน

with open(raw_path, "w", encoding="utf-8") as f:
    f.write(raw_text)

raw_path

'pandas_data/raw_sales.csv'

ลองเปิดดูไฟล์ดิบ (raw) เพื่อเห็นปัญหา

In [4]:
with open(raw_path, "r", encoding="utf-8") as f:
    print(f.read())

date,store_id,sales,comment
2025-01-01,001,1200,ok
2025-01-02,001,,
2025-01-03,002,1500,good
2025-01-04,002,1,500,bad_comma
2025-01-05,003,900,ok



ไฟล์ด้านบนมีปัญหา “comma ในตัวเลข” ทำให้คอลัมน์เลื่อน  
นี่คือเหตุผลที่ Data Engineer ต้องเข้าใจพารามิเตอร์ของ `read_csv` และต้องทำ data cleaning

เราจะสร้างไฟล์ที่อ่านได้ปกติ เพื่อสอน `pd.read_csv` แบบเป็นขั้นตอนก่อน  
แล้วค่อยกลับมาดูเคสที่พังในหัวข้อ troubleshooting

In [5]:
clean_path = os.path.join("pandas_data", "sales.csv")

clean_text = """date,store_id,sales,comment
2025-01-01,001,1200,ok
2025-01-02,001,,missing_sales
2025-01-03,002,1500,good
2025-01-04,002,1500,ok
2025-01-05,003,900,ok
"""

with open(clean_path, "w", encoding="utf-8") as f:
    f.write(clean_text)

clean_path

'pandas_data/sales.csv'

## 3) `pd.read_csv()` แบบพื้นฐาน

✅ ลองอ่านไฟล์ แล้วดูตารางข้อมูล


In [6]:
df = pd.read_csv(clean_path)
df

,date,store_id,sales,comment
0,2025-01-01,1,1200.0,ok
1,2025-01-02,1,NaN,missing_sales
2,2025-01-03,2,1500.0,good
3,2025-01-04,2,1500.0,ok
4,2025-01-05,3,900.0,ok


## 4) ตรวจสอบข้อมูลเบื้องต้น (Data Engineer Checklist)

สิ่งที่ควรเช็คทันทีหลังอ่านไฟล์:
- ขนาดข้อมูล: `df.shape`
- ชื่อคอลัมน์: `df.columns`
- ชนิดข้อมูล: `df.dtypes`
- ตัวอย่างข้อมูล: `df.head()`
- ค่าว่าง: `df.isna().sum()`

✅ ลองรันทีละ cell

In [7]:
df.shape

(5, 4)

In [8]:
df.columns

Index(['date', 'store_id', 'sales', 'comment'], dtype='str')

In [9]:
df.dtypes

date            str
store_id      int64
sales       float64
comment         str
dtype: object

In [10]:
df.head()

,date,store_id,sales,comment
0,2025-01-01,1,1200.0,ok
1,2025-01-02,1,NaN,missing_sales
2,2025-01-03,2,1500.0,good
3,2025-01-04,2,1500.0,ok
4,2025-01-05,3,900.0,ok


In [11]:
df.isna().sum()

date        0
store_id    0
sales       1
comment     0
dtype: int64

## 5) พารามิเตอร์สำคัญของ `pd.read_csv()` (DE Focus)

`pd.read_csv()` มีพารามิเตอร์เยอะ เพราะต้องรองรับไฟล์จากหลายระบบ  
ด้านล่างคือพารามิเตอร์ที่ Data Engineer เจอบ่อย (ควรรู้จักชื่อและใช้ได้)

---

### 5.1 `encoding`
กำหนด encoding ของไฟล์ (เหมือนที่เราใช้กับ `open`)

ตัวอย่าง:
- `encoding="utf-8"` (แนะนำ)
- `encoding="utf-8-sig"` (ไฟล์จาก Excel บางกรณี)
- `encoding="cp874"` (Windows ไทยเก่า)

✅ ทดลอง (ไฟล์เราคือ utf-8)

In [12]:
df_utf8 = pd.read_csv(clean_path, encoding="utf-8")
df_utf8.head()

,date,store_id,sales,comment
0,2025-01-01,1,1200.0,ok
1,2025-01-02,1,NaN,missing_sales
2,2025-01-03,2,1500.0,good
3,2025-01-04,2,1500.0,ok
4,2025-01-05,3,900.0,ok


### 5.2 `sep` หรือ `delimiter`
กำหนดตัวคั่นคอลัมน์
- CSV ส่วนใหญ่ใช้ `,`
- บางไฟล์ใช้ `;` หรือ `\t` (TSV)

ตัวอย่างอ่านไฟล์ TSV:
```python
pd.read_csv("file.tsv", sep="\t")
```

### 5.3 `header` และ `names`
- `header=0` หมายถึงบรรทัดแรกเป็น header (default)
- ถ้าไฟล์ไม่มี header ให้ใช้ `header=None` แล้วกำหนด `names=[...]`

✅ ทดลองจำลองไฟล์ “ไม่มี header”

In [14]:
no_header_path = os.path.join("pandas_data", "sales_no_header.csv")

no_header_text = """2025-01-01,001,1200,ok
2025-01-02,001,,missing_sales
"""

with open(no_header_path, "w", encoding="utf-8") as f:
    f.write(no_header_text)

df_no_header = pd.read_csv(no_header_path)
df_no_header

,2025-01-01,001,1200,ok
0,2025-01-02,1,NaN,missing_sales


In [15]:
df_no_header = pd.read_csv(no_header_path, header=None, names=["date", "store_id", "sales", "comment"])
df_no_header

,date,store_id,sales,comment
0,2025-01-01,1,1200.0,ok
1,2025-01-02,1,NaN,missing_sales


### 5.4 `dtype`
กำหนดชนิดข้อมูลของคอลัมน์ตั้งแต่ตอนอ่านไฟล์ (สำคัญมากในงาน DE)

ปัญหาที่เจอบ่อย:
- store_id เช่น 001 ถ้าอ่านเป็น int → จะกลายเป็น 1 (leading zero หาย)
- คอลัมน์ตัวเลขปนข้อความ → pandas อาจอ่านเป็น object

✅ กำหนด `store_id` เป็น string เพื่อรักษา leading zero

In [16]:
df_typed = pd.read_csv(clean_path, dtype={"store_id": "string"})
df_typed.dtypes

date            str
store_id     string
sales       float64
comment         str
dtype: object

### 5.5 `parse_dates`
แปลงคอลัมน์วันที่ให้เป็น datetime ตั้งแต่ตอนอ่านไฟล์

✅ ทดลอง

In [17]:
df_dates = pd.read_csv(clean_path, parse_dates=["date"])
df_dates.dtypes

date        datetime64[us]
store_id             int64
sales              float64
comment                str
dtype: object

### 5.6 `na_values`
กำหนดค่าที่ถือว่าเป็น missing เช่น `""`, `"NA"`, `"null"`

ในงานจริง บางระบบใช้ค่าแปลก ๆ เช่น `"N/A"`, `"-"`

✅ ทดลองสร้างไฟล์ที่มีค่า `NA`

In [25]:
na_path = os.path.join("pandas_data", "sales_na.csv")
na_text = """date,store_id,sales,comment
2025-01-01,001,1200,ok
2025-01-02,001,,missing_sales
"""
with open(na_path, "w", encoding="utf-8") as f:
    f.write(na_text)

df_na = pd.read_csv(na_path)
df_na

,date,store_id,sales,comment
0,2025-01-01,1,1200.0,ok
1,2025-01-02,1,NaN,missing_sales


In [26]:
df_na = pd.read_csv(na_path, na_values=["NA"])
df_na

,date,store_id,sales,comment
0,2025-01-01,1,1200.0,ok
1,2025-01-02,1,NaN,missing_sales


In [27]:
df_na.isna().sum()

date        0
store_id    0
sales       1
comment     0
dtype: int64

### 5.7 `usecols`
เลือกอ่านเฉพาะบางคอลัมน์ (ช่วยลดเวลาและหน่วยความจำ)

✅ ทดลองอ่านเฉพาะ date กับ sales

In [28]:
df_usecols = pd.read_csv(clean_path, usecols=["date", "sales"])
df_usecols.head()

,date,sales
0,2025-01-01,1200.0
1,2025-01-02,NaN
2,2025-01-03,1500.0
3,2025-01-04,1500.0
4,2025-01-05,900.0


### 5.8 `nrows` และ `skiprows`
- `nrows` อ่านแค่บางแถว (ทดสอบไฟล์ใหญ่)
- `skiprows` ข้ามแถวบน ๆ (บางไฟล์มี metadata ก่อน header)

✅ ทดลอง

In [29]:
pd.read_csv(clean_path, nrows=2)

,date,store_id,sales,comment
0,2025-01-01,1,1200.0,ok
1,2025-01-02,1,NaN,missing_sales


### 5.9 `chunksize` (สำคัญสำหรับไฟล์ใหญ่)

ถ้าไฟล์ใหญ่มากจนอ่านทีเดียวไม่ไหว  
เราสามารถอ่านเป็น “ชิ้น” (chunk) ได้

ตัวอย่าง:
```python
for chunk in pd.read_csv("big.csv", chunksize=100000):
    ทำอะไรกับ chunk
```

ใน notebook นี้เราจะลองกับไฟล์เล็กเพื่อให้เห็นรูปแบบ

In [30]:
chunks = pd.read_csv(clean_path, chunksize=2)

for i, chunk in enumerate(chunks, start=1):
    print("Chunk", i)
    print(chunk)
    print("---")

Chunk 1
         date  store_id   sales        comment
0  2025-01-01         1  1200.0             ok
1  2025-01-02         1     NaN  missing_sales
---
Chunk 2
         date  store_id  sales comment
2  2025-01-03         2   1500    good
3  2025-01-04         2   1500      ok
---
Chunk 3
         date  store_id  sales comment
4  2025-01-05         3    900      ok
---


## 6) Transformation เบื้องต้นที่ใช้ใน pipeline

ตัวอย่างที่ Data Engineer ทำบ่อย:
- เติมค่า missing (`fillna`)
- แปลงชนิดข้อมูล (`astype`)
- สร้างคอลัมน์ใหม่ (derived column)

In [31]:
df2 = pd.read_csv(clean_path, dtype={"store_id": "string"})

# เติมยอดขายที่หายเป็น 0 (ขึ้นกับ business rule)
df2["sales"] = df2["sales"].fillna(0)

# สร้างคอลัมน์ flag ว่ายอดขายขาดหายหรือไม่
df2["sales_missing"] = df2["comment"].eq("missing_sales")

df2

,date,store_id,sales,comment,sales_missing
0,2025-01-01,001,1200.0,ok,False
1,2025-01-02,001,0.0,missing_sales,True
2,2025-01-03,002,1500.0,good,False
3,2025-01-04,002,1500.0,ok,False
4,2025-01-05,003,900.0,ok,False


## 7) เขียนไฟล์ออกด้วย `to_csv()` (DE Focus)

พารามิเตอร์ที่ควรรู้:
- `index=False` (ส่วนใหญ่ไม่อยากเขียน index ลงไฟล์)
- `encoding="utf-8"` หรือ `"utf-8-sig"` (ให้ Excel เปิดไทยได้ดีขึ้น)
- `na_rep` แทนค่า missing ตอนเขียนไฟล์
- `sep` ถ้าต้องการ delimiter อื่น

✅ ทดลองเขียนไฟล์ผลลัพธ์


In [32]:
out_path = os.path.join("pandas_data", "sales_cleaned.csv")

df2.to_csv(out_path, index=False, encoding="utf-8", na_rep="")

out_path

'pandas_data/sales_cleaned.csv'

ตรวจสอบไฟล์ที่เขียนออกมา

In [33]:
with open(out_path, "r", encoding="utf-8") as f:
    print(f.read())

date,store_id,sales,comment,sales_missing
2025-01-01,001,1200.0,ok,False
2025-01-02,001,0.0,missing_sales,True
2025-01-03,002,1500.0,good,False
2025-01-04,002,1500.0,ok,False
2025-01-05,003,900.0,ok,False



## 8) Mini Pipeline: อ่านหลายไฟล์ → รวม → ทำความสะอาด → เขียนผลลัพธ์

Data Engineer มักได้ไฟล์รายวันหลายไฟล์  
เราจะจำลอง 2 ไฟล์ แล้วรวมเป็นไฟล์เดียว

✅ ขั้นตอน:
1) สร้างไฟล์ 2 ไฟล์
2) อ่านทั้งหมดจากโฟลเดอร์
3) รวม (concat)
4) ทำความสะอาด
5) เขียนไฟล์รวม

In [34]:
import glob

# สร้างไฟล์รายวันจำลอง
daily1 = os.path.join("pandas_data", "daily_01.csv")
daily2 = os.path.join("pandas_data", "daily_02.csv")

with open(daily1, "w", encoding="utf-8") as f:
    f.write("date,store_id,sales\n2025-01-01,001,1200\n2025-01-01,002,1500\n")

with open(daily2, "w", encoding="utf-8") as f:
    f.write("date,store_id,sales\n2025-01-02,001,\n2025-01-02,003,900\n")

# 1) หาไฟล์ทั้งหมด
paths = sorted(glob.glob(os.path.join("pandas_data", "daily_*.csv")))
paths

['pandas_data/daily_01.csv', 'pandas_data/daily_02.csv']

In [35]:
# 2) อ่านและรวม
dfs = [pd.read_csv(p, dtype={"store_id": "string"}, parse_dates=["date"]) for p in paths]
all_df = pd.concat(dfs, ignore_index=True)

# 3) ทำความสะอาด
all_df["sales"] = all_df["sales"].fillna(0)

all_df

,date,store_id,sales
0,2025-01-01,001,1200.0
1,2025-01-01,002,1500.0
2,2025-01-02,001,0.0
3,2025-01-02,003,900.0


In [36]:
# 4) เขียนไฟล์รวม
merged_path = os.path.join("pandas_data", "daily_merged.csv")
all_df.to_csv(merged_path, index=False, encoding="utf-8")

merged_path

'pandas_data/daily_merged.csv'

## 9) Troubleshooting ที่เจอบ่อยใน `read_csv`

### กรณี 1: parser error / คอลัมน์ไม่เท่ากัน
สาเหตุ:
- ข้อมูลมี comma แฝง เช่น `1,500` โดยไม่ได้ใส่ quote `"1,500"`

แนวทางแก้ (ระดับเบื้องต้น):
- ตรวจไฟล์ดิบก่อน
- ใช้ `thousands=","` (ในบางกรณี) หรือทำ preprocessing
- ถ้าหนักมาก: ต้องทำขั้นตอนทำความสะอาดก่อนอ่าน หรือใช้ engine/กฎพิเศษ

### กรณี 2: ตัวอักษรไทยอ่านเป็นภาษาต่างดาว
สาเหตุ:
- encoding ไม่ตรง
แนวทางแก้:
- ลอง `encoding="utf-8-sig"` หรือ `encoding="cp874"` ตามแหล่งไฟล์

### กรณี 3: store_id leading zero หาย
แนวทางแก้:
- ตั้ง `dtype={"store_id": "string"}` ตั้งแต่ตอนอ่าน

## 10) แบบฝึกหัด (DE-style)

ให้ทำตามนี้:
1) อ่านไฟล์ `daily_merged.csv` ที่เราสร้างไว้
2) ตั้งค่า `dtype` ให้ `store_id` เป็น string และ `parse_dates` สำหรับ date
3) เติมค่า sales ที่หายเป็น 0
4) เขียนไฟล์ออกชื่อ `daily_final.csv` โดย `index=False` และ `encoding="utf-8"`

✅ ทำใน cell ด้านล่าง

In [37]:
# TODO: ทำตามโจทย์
merged_path = os.path.join("pandas_data", "daily_merged.csv")

df_final = pd.read_csv(merged_path, dtype={"store_id": "string"}, parse_dates=["date"])
df_final["sales"] = df_final["sales"].fillna(0)

final_path = os.path.join("pandas_data", "daily_final.csv")
df_final.to_csv(final_path, index=False, encoding="utf-8")

final_path

'pandas_data/daily_final.csv'

## 11) สรุป (Checklist)

หลังจบบทนี้ คุณควรทำได้:
- ✅ อ่าน CSV ด้วย pandas และตรวจสอบข้อมูลเบื้องต้นได้
- ✅ เข้าใจพารามิเตอร์สำคัญของ `pd.read_csv()` (encoding/sep/header/names/dtype/parse_dates/na_values/usecols/nrows/skiprows/chunksize)
- ✅ ทำ data quality check แบบเร็ว ๆ (shape/dtypes/missing)
- ✅ ทำ transformation ที่ใช้บ่อยใน pipeline (fillna/astype/สร้างคอลัมน์)
- ✅ เขียนไฟล์ด้วย `to_csv()` พร้อมตั้งค่าที่เหมาะสม (index/encoding/na_rep/sep)
- ✅ ทำ mini pipeline อ่านหลายไฟล์ → รวม → ทำความสะอาด → เขียนผลลัพธ์ได้

➡️ บทถัดไป: **numpy** เพื่อเข้าใจ array, performance, และการคำนวณเชิงตัวเลขที่ต่อยอดไปงาน data ได้